In [13]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

#files_path
log_files = "../data/raw/HDFS/HDFS.log"
anomaly_csv = "../data/raw/HDFS/preprocessed/anomaly_label.csv"

#สร้างตัว miner template
template_miner = TemplateMiner()

#text ทดสอบ
log_text = "Receiving block blk_123 src: /10.0.0.1 dest: /10.0.0.2"

#โยนข้อความให้ miner ประมวลผล
result = template_miner.add_log_message(log_text)

#แสดง output
print(result)
print("Cluster ID (Event ID):", result["cluster_id"])
print("Template ที่ได้:", result["template_mined"])

config file not found: drain3.ini


{'change_type': 'cluster_created', 'cluster_id': 1, 'cluster_size': 1, 'template_mined': 'Receiving block blk_123 src: /10.0.0.1 dest: /10.0.0.2', 'cluster_count': 1}
Cluster ID (Event ID): 1
Template ที่ได้: Receiving block blk_123 src: /10.0.0.1 dest: /10.0.0.2


In [14]:
#ทดสอบการทำงานของ drain3 (ทำการจับ template เองโดยอัตโนมัติโดยไม่ต้องเขียน regex อะไรต่าง = <*>, อะไรเหมือน = template)

#log text ทดสอบ 2
log_text2 = "Receiving block blk_999 src: /192.168.1.1 dest: /192.168.1.2"
result = template_miner.add_log_message(log_text2)

print(result)

{'change_type': 'cluster_template_changed', 'cluster_id': 1, 'cluster_size': 2, 'template_mined': 'Receiving block <*> src: <*> dest: <*>', 'cluster_count': 1}


In [12]:
import re

print("กำลังอ่าน 5 บรรทัดแรก...\n")

#อ่านไฟล์ HDFS
with open(log_files, 'r') as log:
    for i, line in enumerate(log):
        if i >=5: # 5 บรรทัดแรก ปรับเอาตรงนี้ได้
            break
        line = line.strip()

        #แยก header กับ content ด้วยเครื่องหมาย : แรกที่เจอ
        part = line.split(': ',1)
        if len(part) == 2:
            header, content = part[0], part[1] # ท่อนหน้า = header, ท่อนหลัง = content
        else:
            content = line

        #ส่ง content ให้ drain3 ทำการวิเคราะห์
        result = template_miner.add_log_message(content) 
        
        print(line)
        print(f"บรรทัดที่ {i+1}:")
        print(f" ข้อความดิบ: {content}")
        print(f" Header: {header}")
        print(f" -> Event ID: E{result['cluster_id']}")
        print(f" -> Template: {result['template_mined']}")
        print(f"."*60)

กำลังอ่าน 5 บรรทัดแรก...

081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
บรรทัดที่ 1:
 ข้อความดิบ: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
 Header: 081109 203518 143 INFO dfs.DataNode$DataXceiver
 -> Event ID: E1
 -> Template: Receiving block <*> src: <*> dest: <*>
............................................................
081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
บรรทัดที่ 2:
 ข้อความดิบ: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
 Header: 081109 203518 35 INFO dfs.FSNamesystem
 -> Event ID: E2
 -> Template: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
.................................

In [32]:
#จัด header แล้วรวมกับ content
from collections import defaultdict
import re

#กล่องสำหรับเก็บ session: block id [list ของ event id]
sessions = defaultdict(list)

#regex สำหรับ block id (blk_ตามด้วยตัวเลข หรืออาจมีเครื่องหมายลบ)
block_regex = re.compile(r'(blk_-?\d+)')

# อ่านไฟล์ sample 2k ทั้งหมด (2,000 บรรทัด)
log_path = '../data/raw/HDFS/HDFS_2k.log'

with open (log_path, 'r') as log:
    for line in log:
        line = line.strip()

        #ดึง block id ออกจากบรรทัด
        block_match = block_regex.search(line)
        if not block_match:
            continue #ถ้าไม่มี block_match(สำหรับตอนนี้ regex คือ blk_*) ให้ข้ามไป
        block_id = block_match.group(1)

        #แยกแค่ content ส่งให้ drain3
        parts = line.split(':', 1)
        content = parts[1] if len(parts) == 2 else line

        #Parse ด้วย drain3 เพื่อเอา event id
        result = template_miner.add_log_message(content)
        event_id = f"E{result['cluster_id']}"

        #เอา event id ใส่ใน session ของ block id นั้นๆ
        sessions[block_id].append(event_id)

print(f"ประมวผลเสร็จสิ้น")
print(f"จำนวนบรรทัดที่อ่าน: 2000")
print(f"จำนวน session (block id) ที่ได้: {len(sessions)} sessions")
print(f"จำนวน event type (Template) ทั้งหมด: {len(template_miner.drain.id_to_cluster)} แบบ\n")

print(f"="*60)
print(f"ตัวอย่าง 3 ลำดับของ sessions แรก")
for i,(blk,seq) in enumerate(sessions.items()):
    if i >=3:
        break
print(f"Session {i+1} [{blk}]:")
print(f"   ความยาว: {len(seq)} เหตุการณ์")
print(f"   ลำดับเหตุการณ์: {seq}")
print(f"="*60)

ประมวผลเสร็จสิ้น
จำนวนบรรทัดที่อ่าน: 2000
จำนวน session (block id) ที่ได้: 1994 sessions
จำนวน event type (Template) ทั้งหมด: 51 แบบ

ตัวอย่าง 3 ลำดับของ sessions แรก
Session 4 [blk_8229193803249955061]:
   ความยาว: 1 เหตุการณ์
   ลำดับเหตุการณ์: ['E3']


In [49]:
#ผูกเฉลยเข้ากับการตรวจจับ
import pandas as pd

#โหลดไฟล์เฉลย
labels_path = '../data/raw/HDFS/preprocessed/anomaly_label.csv'
df_labels = pd.read_csv(labels_path)

#แปลง labels จากตัวอักษรเป็น normal = 0, anomaly = 1
label_mapping = {'Normal':0, 'Anomaly':1}
df_labels['Label'] = df_labels['Label'].map(label_mapping)

#แปลง dict session เป็น dataframe เพื่อนำเสนอตาราง
df_sessions = pd.DataFrame([
    {'BlockId': blk, 'Sequence': seq}
    for blk, seq in sessions.items()
])

#รวม (Merge/Join) ด้วย BlockID เหมือนทำ Inner Join
df_dataset = pd.merge(df_sessions, df_labels, on='BlockId', how='inner')

#ดูผลลัพธ์
print(f"ได้ข้อมูลพร้อมใช้งานทั้งหมด: {len(df_dataset):,}sessions\n")
print(f"=== ตัวอย่าง 5 แถวแรกของชุดข้อมูล ===")
print(df_dataset)

#check count สำหรับ normal, anomaly
print(f"\n=== เช็คจำนวน ===")
print(df_dataset['Label'].value_counts())

ได้ข้อมูลพร้อมใช้งานทั้งหมด: 1,994sessions

=== ตัวอย่าง 5 แถวแรกของชุดข้อมูล ===
                       BlockId Sequence  Label
0        blk_38865049064139660     [E3]      0
1     blk_-6952295868487656571     [E3]      0
2      blk_7128370237687728475     [E5]      0
3      blk_8229193803249955061     [E3]      0
4     blk_-6670958622368987959     [E3]      0
...                        ...      ...    ...
1989   blk_4198733391373026104     [E1]      0
1990  blk_-5815145248455404269     [E4]      0
1991   blk_-295306975763175640     [E1]      0
1992   blk_5225719677049010638     [E3]      0
1993   blk_4343207286455274569     [E1]      0

[1994 rows x 3 columns]

=== เช็คจำนวน ===
Label
0    1926
1      68
Name: count, dtype: int64


In [58]:
#แบ่งไฟล์ข้อมูลสำหรับการทำ Train/Val/Test
from sklearn.model_selection import train_test_split
import os

#แปลง list ของ sequence ที่คั่นด้วย , ให้เป็นช่องว่างเพื่อให้บันทึกลง csv ง่ายๆ
df_export = df_dataset.copy()
df_export['Sequence'] = df_export['Sequence'].apply(lambda seq: ' '.join(seq))

#แบ่ง train ออกมา 70% เก็บอีก 30% สำหรับ validate, test
train_df, temp_df = train_test_split(
    df_export,
    test_size=0.30,
    random_state=42, #ใส่ seed เพื่อให้รัยออกมาเหมือนเดิมทุกครั้ง
    stratify=df_export['Label'] #บังคับให้สัดส่วน Anomaly เท่ากัน
)

#แบ่งรอบที่ 2 เอา 30% มาแบ่งครึ่งเพื่อทำ validate, test
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['Label']
)

#สรุปขนาดของแต่ละชุดข้อมูล
output_dir = "../data/processed/HDFS"
os.makedirs(output_dir, exist_ok=True)

train_df.to_csv(f'{output_dir}/train.csv',index=False)
val_df.to_csv(f'{output_dir}/val.csv',index=False)
test_df.to_csv(f'{output_dir}/test.csv',index=False)

print(f"บันทึกไฟล์เรียบร้อย ที่ตำแหน่ง: {output_dir}/")
print(f" - train.csv")
print(f" - val.csv")
print(f" - test.csv")

บันทึกไฟล์เรียบร้อย ที่ตำแหน่ง: ../data/processed/HDFS/
 - train.csv
 - val.csv
 - test.csv
